# Grade 12 · Unit 2 — Regression and classification

> **SYNTHETIC** health table.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

h = pd.read_csv(data_file("health-synthetic", "health_synthetic.csv"))
FEATURES = ["antenatal_visits", "wealth_quintile", "mother_age"]
X, y = h[FEATURES], h["institutional_delivery"]
print(h.shape, "| share delivering in a facility:", round(float(y.mean()), 3))

## Regression: predict birth weight (a number)

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
Xr = h[["wealth_quintile", "antenatal_visits"]]
lin = LinearRegression().fit(Xr, h["birth_weight_kg"])
# the same numbers as the least-squares solution in Grade 11:
A = np.column_stack([np.ones(len(h)), Xr.to_numpy()])
beta, *_ = np.linalg.lstsq(A, h["birth_weight_kg"].to_numpy(), rcond=None)
assert np.allclose([lin.intercept_, *lin.coef_], beta)
print("intercept", round(lin.intercept_, 3), "| coefficients", dict(zip(Xr.columns, lin.coef_.round(3))))

## Classification: predict facility delivery (yes/no)
**Predict:** will probability rise or fall with wealth?

In [ ]:
logit = LogisticRegression().fit(X, y)
probs = logit.predict_proba(X)[:, 1]
assert probs.min() >= 0 and probs.max() <= 1
print(dict(zip(FEATURES, logit.coef_[0].round(2))))
poor = pd.DataFrame({"antenatal_visits": [2], "wealth_quintile": [1], "mother_age": [25]})
rich = pd.DataFrame({"antenatal_visits": [6], "wealth_quintile": [5], "mother_age": [25]})
print("P(facility) poor/2 visits:", round(float(logit.predict_proba(poor)[0, 1]), 2), "| rich/6 visits:", round(float(logit.predict_proba(rich)[0, 1]), 2))

## Threshold = a decision
A probability becomes a yes/no only when we choose a threshold.

In [ ]:
for t in (0.3, 0.5, 0.7):
    print(t, "→ predicted yes for", round(float((probs >= t).mean()), 3), "of records")

**Make:** explain to a health-post worker, in two sentences, what the probability means and what it does not.